# 01 · Análisis Exploratorio — Generación renovable en Argentina (2011–2026)

**Fuente:** Base de datos de Energía Renovable (generación mensual por máquina, en GWh), versión 2026-09.

## Pregunta guía
> ¿Se puede predecir la generación mensual de una central renovable a partir de su fuente, ubicación, programa de licitación y estacionalidad? ¿Qué tan bien funcionaría según la región?

## Qué hace este notebook
1. Carga y describe el dataset.
2. Detecta y corrige problemas de calidad (filas de demanda mezcladas, duplicados, ceros ambiguos, nombres inconsistentes).
3. Explora la evolución, distribución, geografía y estacionalidad de la generación.
4. Documenta **hallazgos y decisiones** que condicionan el modelado (notebook 02).

> Colocá el Excel original en la carpeta `data/` del proyecto antes de ejecutar.

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RUTA = next(itertools.chain(Path("../data").glob("*Renovables*.xlsx"),
                            Path("data").glob("*Renovables*.xlsx")), None)
assert RUTA is not None, "No encuentro el Excel: copialo en la carpeta data/ del proyecto."
print("Archivo:", RUTA)

## 1. Carga de datos
El Excel tiene 4 hojas. Solo **"Base de Datos"** es la fuente primaria; las otras son tablas resumen calculadas con fórmulas a partir de ella, así que no se usan para modelar.
La primera fila de la hoja está vacía, por eso `header=1`.

In [ ]:
df_raw = pd.read_excel(RUTA, sheet_name="Base de Datos", header=1).dropna(how="all")
print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
print("Nulos por columna:")
print(df_raw.isna().sum())
print("\nRango de fechas:", df_raw["MES"].min().date(), "→", df_raw["MES"].max().date())

## 2. Limpieza y control de calidad

### 2.1 Nombres de columnas y texto
Hay espacios sobrantes (por ejemplo `" Demanda MEM"`), mayúsculas/minúsculas mezcladas y la columna `AÑO` es una **fórmula de Excel** (`=YEAR(MES)`). La recalculo desde `MES`.

In [ ]:
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={
    "CENTRAL": "central", "CENTRAL DESCRIPCIÓN": "central_desc", "MAQUINA": "maquina",
    "FUENTE DE ENERGÍA": "fuente", "REGIÓN": "region", "PROVINCIA": "provincia",
    "MES": "mes", "ENERGÍA GENERADA [GWh]": "gwh", "Nueva Generación": "programa",
}).drop(columns=["AÑO"])

for c in ["central", "central_desc", "maquina", "fuente", "region", "provincia", "programa"]:
    df[c] = df[c].astype(str).str.strip().str.upper()

df["mes"] = pd.to_datetime(df["mes"])
df["anio"] = df["mes"].dt.year
df["mes_num"] = df["mes"].dt.month

print("Programas:", sorted(df["programa"].unique()))
print("Fuentes:  ", sorted(df["fuente"].unique()))

**Observación:** aparece un programa llamado `RENOVAR 202` (probablemente truncado, ¿RENOVAR 2021?). No puedo confirmarlo desde los datos, así que **lo dejo como categoría separada** y lo anoto como limitación en vez de asumir una fusión.

### 2.2 Filas de demanda mezcladas con generación
La fuente `DEMANDA MEM` no es generación renovable: es la demanda total del sistema (miles de GWh por mes). Si se deja mezclada, distorsiona cualquier estadística. La separo como serie de contexto.

In [ ]:
es_demanda = df["fuente"] == "DEMANDA MEM"
print(f"Filas de demanda: {es_demanda.sum()}  |  Filas de generación: {(~es_demanda).sum()}")

df_dem = (df[es_demanda].groupby("mes", as_index=False)["gwh"].sum()
          .rename(columns={"gwh": "demanda_gwh"}))
df_ren = df[~es_demanda].copy()

print("\nMáximo con la demanda mezclada :", f"{df['gwh'].max():,.0f} GWh")
print("Máximo solo generación renovable:", f"{df_ren['gwh'].max():,.1f} GWh")

### 2.3 Duplicados (máquina + mes)

In [ ]:
clave = ["maquina", "mes"]
dups = df_ren[df_ren.duplicated(clave, keep=False)].sort_values(clave)
display(dups[["maquina", "fuente", "provincia", "mes", "gwh", "programa"]])

Hay 5 pares repetidos. Uno es un duplicado exacto (`CASAEO`); en los otros, las dos filas tienen valores distintos (en varios casos una vale cero y la otra tiene generación, lo que parece un registro corregido o parcial).
**Decisión:** eliminar duplicados exactos y, cuando la misma máquina-mes tiene valores distintos, **conservar el mayor**. Es una suposición razonable pero no verificable con la fuente, así que queda documentada.

In [ ]:
n0 = len(df_ren)
df_ren = df_ren.drop_duplicates()
df_ren = df_ren.sort_values("gwh").drop_duplicates(clave, keep="last").sort_values(clave).reset_index(drop=True)
print(f"Filas antes: {n0:,}  →  después: {len(df_ren):,}")

### 2.4 Ceros: ¿central que aún no existe o parada real?
Cerca del 10% de las filas valen 0. Tratarlos todos igual sería un error:
- **Ceros de pre-operación:** anteriores al primer mes con generación de esa máquina (la central todavía no operaba).
- **Ceros operativos:** posteriores al arranque (parada, mantenimiento, falta de recurso, falla o dato faltante).

In [ ]:
primer_mes = df_ren[df_ren["gwh"] > 0].groupby("maquina")["mes"].min().rename("primer_mes_gen")
df_ren = df_ren.join(primer_mes, on="maquina")

cero = df_ren["gwh"] == 0
pre_op = cero & (df_ren["primer_mes_gen"].isna() | (df_ren["mes"] < df_ren["primer_mes_gen"]))
df_ren["cero_pre_operacion"] = pre_op
df_ren["cero_operativo"] = cero & ~pre_op

resumen = pd.Series({
    "Filas totales": len(df_ren),
    "Ceros totales": int(cero.sum()),
    "  · pre-operación": int(pre_op.sum()),
    "  · operativos": int((cero & ~pre_op).sum()),
    "Máquinas que nunca generaron": df_ren.loc[df_ren["primer_mes_gen"].isna(), "maquina"].nunique(),
})
print(resumen.to_string(float_format=lambda x: f"{x:,.0f}"))

**Decisión:** excluyo los ceros de pre-operación del análisis y del modelado (no representan generación, la central no existía). Mantengo los ceros operativos, porque son parte de la realidad que el modelo debe aprender (paradas, falta de recurso).

In [ ]:
df_model = df_ren[~df_ren["cero_pre_operacion"]].copy()
print(f"Dataset para análisis/modelado: {df_model.shape[0]:,} filas · {df_model['maquina'].nunique()} máquinas")

df_model.to_csv(RUTA.parent / "renovables_limpio.csv", index=False)
df_dem.to_csv(RUTA.parent / "demanda_mem.csv", index=False)
print("Guardados en:", RUTA.parent)

## 3. Exploración
### 3.1 Cobertura: un panel desbalanceado
Las máquinas entran en operación en distintos momentos. Eso significa que el parque **crece** a lo largo del tiempo y que las series tienen largos muy distintos.

In [ ]:
maq_anio = df_model.groupby("anio")["maquina"].nunique()
ax = maq_anio.plot(kind="bar", color="#3b7dd8", figsize=(10, 4))
ax.set(title="Máquinas en operación por año (2026 = enero a septiembre)", xlabel="", ylabel="N° de máquinas")
plt.show()

largo = df_model.groupby("maquina").size()
print("Meses de historia por máquina:")
print(largo.describe().round(1).to_string())

### 3.2 Evolución de la generación por fuente
Se excluye 2026 porque solo tiene 9 meses y no es comparable.

In [ ]:
anual = (df_model[df_model["anio"] < 2026]
         .pivot_table(index="anio", columns="fuente", values="gwh", aggfunc="sum"))
anual.plot(kind="bar", stacked=True, figsize=(11, 5), colormap="viridis")
plt.title("Generación renovable anual por fuente [GWh]")
plt.ylabel("GWh"); plt.xlabel("")
plt.legend(title="Fuente", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

total_fuente = df_model.groupby("fuente")["gwh"].sum().sort_values(ascending=False)
print((total_fuente / total_fuente.sum() * 100).round(1).rename("% del total histórico").to_string())

### 3.3 ¿Cuánta de la demanda cubren estas renovables?
Comparo la generación renovable mensual (de este dataset) contra la demanda del MEM. Es una aproximación: solo cuenta las centrales incluidas en esta base (no incluye hidro grande).

In [ ]:
mensual = (df_model.groupby("mes")["gwh"].sum().rename("renovable_gwh").to_frame()
           .join(df_dem.set_index("mes")))
mensual["participacion_pct"] = mensual["renovable_gwh"] / mensual["demanda_gwh"] * 100

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(mensual.index, mensual["participacion_pct"], alpha=0.35, label="Mensual")
ax.plot(mensual.index, mensual["participacion_pct"].rolling(12).mean(), lw=2.5, label="Media móvil 12 meses")
ax.set(title="Participación de estas renovables sobre la demanda del MEM", ylabel="% de la demanda")
ax.legend(); plt.show()

print(mensual.groupby(mensual.index.year)["participacion_pct"].mean().round(1).rename("% medio anual").to_string())

### 3.4 Distribución de la generación y outliers
La generación por máquina es muy asimétrica: muchas máquinas chicas y pocos parques grandes. Por eso se usa escala logarítmica.

In [ ]:
pos = df_model[df_model["gwh"] > 0]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
sns.histplot(np.log10(pos["gwh"]), bins=50, ax=ax[0], color="#3b7dd8")
ax[0].set(title="Distribución (log10 de GWh mensuales)", xlabel="log10(GWh)")
sns.boxplot(data=pos, x="fuente", y="gwh", ax=ax[1])
ax[1].set_yscale("log"); ax[1].set(title="GWh mensuales por fuente (escala log)", xlabel="", ylabel="GWh")
ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

print("Estadísticos por fuente (GWh/mes por máquina):")
print(df_model.groupby("fuente")["gwh"].agg(["count", "mean", "median", "max"]).round(2).to_string())

In [ ]:
def pct_outliers(s):
    x = np.log1p(s)
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).mean() * 100

print("% de outliers (regla IQR sobre log1p) por fuente:")
print(df_model.groupby("fuente")["gwh"].apply(pct_outliers).round(1).to_string())

print("\nLos 8 valores más altos:")
display(df_model.nlargest(8, "gwh")[["maquina", "central_desc", "fuente", "provincia", "mes", "gwh"]])

Los valores más altos corresponden a **parques grandes reales**, no a errores de carga. **Decisión:** no eliminar outliers; en su lugar, modelar con transformación logarítmica del objetivo y usar modelos basados en árboles, que son robustos a la escala.

### 3.5 Geografía: ¿dónde se genera y con qué?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
(df_model.groupby("region")["gwh"].sum().sort_values().plot.barh(ax=ax[0], color="#3b7dd8"))
ax[0].set(title="Generación acumulada por región [GWh]", ylabel="")
(df_model.groupby("provincia")["gwh"].sum().nlargest(10).sort_values().plot.barh(ax=ax[1], color="#e07b39"))
ax[1].set(title="Top 10 provincias [GWh]", ylabel="")
plt.tight_layout(); plt.show()

mix = pd.crosstab(df_model["region"], df_model["fuente"], values=df_model["gwh"],
                  aggfunc="sum", normalize="index").fillna(0) * 100
plt.figure(figsize=(9, 4.5))
sns.heatmap(mix, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={"label": "% de la generación regional"})
plt.title("Mix de fuentes dentro de cada región"); plt.ylabel(""); plt.xlabel("")
plt.show()

### 3.6 Programas de licitación (`programa`)

In [ ]:
prog = (df_model.groupby("programa")
        .agg(maquinas=("maquina", "nunique"), gwh_total=("gwh", "sum"))
        .sort_values("gwh_total", ascending=False))
display(prog)

### 3.7 Estacionalidad
Comparar promedios mensuales crudos es engañoso porque el parque crece durante el año. Para aislar la estacionalidad, calculo un **índice**: la generación de cada mes dividida por el promedio de esa misma máquina ese año (1,0 = mes promedio). Solo se usan años completos (2019–2025) y máquina-años con 12 meses.

In [ ]:
op = df_model[df_model["anio"].between(2019, 2025)].copy()
op["media_maq_anio"] = op.groupby(["maquina", "anio"])["gwh"].transform("mean")
op["n_meses"] = op.groupby(["maquina", "anio"])["gwh"].transform("count")
op = op[(op["media_maq_anio"] > 0) & (op["n_meses"] == 12)]
op["indice"] = op["gwh"] / op["media_maq_anio"]

est = op.pivot_table(index="fuente", columns="mes_num", values="indice", aggfunc="mean")
plt.figure(figsize=(11, 4))
sns.heatmap(est, annot=True, fmt=".2f", cmap="RdYlBu_r", center=1.0, cbar_kws={"label": "Índice (1 = promedio)"})
plt.title("Estacionalidad: índice de generación por fuente y mes"); plt.xlabel("Mes"); plt.ylabel("")
plt.show()
display(est.round(2))

### 3.8 Persistencia: ¿cuánto predice el pasado reciente?
Mido la correlación entre la generación de una máquina y la de **1 mes antes** y **12 meses antes**.

In [ ]:
base = df_model[["maquina", "mes", "gwh"]]
l1 = base.assign(mes=base["mes"] + pd.DateOffset(months=1)).rename(columns={"gwh": "lag_1m"})
l12 = base.assign(mes=base["mes"] + pd.DateOffset(months=12)).rename(columns={"gwh": "lag_12m"})
tmp = base.merge(l1, on=["maquina", "mes"]).merge(l12, on=["maquina", "mes"])

logs = np.log1p(tmp[["gwh", "lag_1m", "lag_12m"]])
print(f"Pares máquina-mes con ambos rezagos disponibles: {len(tmp):,}")

print("\n(a) Correlación global (mezcla máquinas grandes y chicas):")
display(logs.corr(method="spearman").round(3))

# (b) Dentro de cada máquina: resto el promedio propio para quitar el efecto "tamaño"
intra = logs - logs.groupby(tmp["maquina"]).transform("mean")
print("(b) Correlación dentro de cada máquina (sin el efecto tamaño):")
display(intra.corr(method="spearman").round(3))

### 3.9 (Opcional) Consulta con DuckDB
El programa incluye análisis local con SQL. Esta celda hace una consulta directa sobre el DataFrame.

In [ ]:
try:
    import duckdb
    display(duckdb.sql("""
        SELECT anio, fuente, ROUND(SUM(gwh), 1) AS gwh_total, COUNT(DISTINCT maquina) AS maquinas
        FROM df_model
        WHERE anio >= 2024
        GROUP BY anio, fuente
        ORDER BY anio DESC, gwh_total DESC
    """).df())
except ImportError:
    print("DuckDB no está instalado (pip install duckdb). Celda opcional.")

## 4. Hallazgos y decisiones
### Hallazgos

1. **Calidad de datos.** La hoja principal mezcla 189 filas de *Demanda MEM* con la generación (inflaban el máximo de ~168 a ~14.000 GWh), tiene 5 pares máquina-mes repetidos y la columna `AÑO` es una fórmula. Después de limpiar quedan ~22.800 filas y 289 máquinas.
2. **Ceros ambiguos.** ~10% de las filas valen cero, pero solo 340 son de pre-operación (la central todavía no existía); los otros ~1.950 son ceros operativos reales (paradas, falta de recurso, datos faltantes).
3. **Crecimiento fuerte y panel desbalanceado.** Las máquinas en operación pasan de unas 50 (2011) a ~280 (2026) y la participación sobre la demanda del MEM sube de ~1-2% a ~19% (promedio 2025). El salto fuerte arranca en 2019, con los programas de licitación. La mediana de historia por máquina es de solo ~73 meses.
4. **Una sola fuente domina.** La eólica aporta ~65% de toda la generación histórica, seguida de solar (~14%) e hidro ≤50MW (~13%).
5. **Región y fuente casi se confunden.** Buenos Aires y Patagonia son ~96-99% eólicas, Litoral ~99% biogás, Noreste ~80% biomasa, Cuyo solar + hidro. Patagonia + Buenos Aires concentran ~56% de la generación.
6. **Estacionalidad según la fuente.** Hidro ≤50MW es la más estacional (índice ~0,5 en junio-julio vs ~1,3 en verano), solar baja en invierno (~0,65 en junio) y biomasa sube en invierno. La eólica es casi plana a nivel mensual (0,89 a 1,09): su variabilidad depende del viento local, que **no está en este dataset**.
7. **Biodiésel es un caso aparte.** Casi toda su generación viene de dos máquinas de Puerto Nuevo durante **12 meses de 2012**; el resto del biodiésel genera casi nada (mediana 0). Eso explica el pico de 2012 en los gráficos y los "outliers" más altos: no es un parque grande, es un episodio puntual y de otra tecnología.
8. **El pasado ayuda, pero menos de lo que parece.** La correlación entre la generación y la del mes anterior es 0,96 en el dato global, pero cae a ~0,64 (1 mes) y ~0,50 (12 meses) cuando se compara cada máquina contra sí misma. La correlación alta global mide sobre todo **tamaño de la central**, no capacidad de predecir.

### Decisiones para el modelado

| Decisión | Motivo |
|---|---|
| Separar `Demanda MEM` de la generación | No es generación; sirve como contexto. |
| Excluir ceros de pre-operación, conservar ceros operativos | Los primeros no son generación; los segundos son parte de lo que hay que aprender. |
| Duplicados con valores distintos: conservar el mayor | Supuesto razonable, **no verificable** con la fuente. |
| No eliminar outliers; usar `log1p(GWh)` y modelos de árboles | Hay mucha asimetría legítima entre centrales chicas y grandes. |
| **Excluir biodiésel del modelo principal** | Un solo episodio (2012), sin patrón aprendible, ~1,4% de la generación. |
| Validación **temporal** (entrenar con años anteriores, testear en los últimos) | Un split aleatorio filtraría información del futuro. |
| Dejar `RENOVAR 202` como categoría aparte | No puedo confirmar a qué programa corresponde. |
| Definir dos escenarios de modelo: **con rezagos** (requiere historial de la máquina) y **sin rezagos** (solo fuente, región, programa y mes) | La API solo podrá predecir para centrales nuevas con el segundo. |

### Limitaciones y sesgos a discutir en el README

- **Sin potencia instalada ni clima:** el modelo no puede separar "central grande" de "buen recurso". Es la mejora más importante: sumar radiación y viento por provincia (por ejemplo NASA POWER).
- **Sesgo de supervivencia geográfico:** solo aparecen centrales que se construyeron, en zonas donde ya se decidió invertir.
- **Confusión región-fuente:** casi no hay eólica en Cuyo ni solar en Patagonia, así que importancias de variables (SHAP) y errores por región se mezclan con los de fuente.
- **Pocos datos en algunas regiones:** Litoral tiene 7 máquinas y Comahue 12; los errores ahí serán inestables.
- **Distribución cambiante:** el parque se multiplicó casi por 6; un modelo entrenado en el pasado puede no extrapolar bien a tecnologías, regiones o regulaciones nuevas.
- **Uso responsable:** no debe usarse para decidir inversiones ni despachos sin validación de especialistas.

### Próximo paso
En `02_modelos.ipynb`: baseline (media por fuente y mes), regresión lineal, Random Forest con Optuna, validación cruzada temporal, error **por región y por fuente**, y SHAP para interpretar.

## 5. Registro de uso de IA copiloto (completar)
La consigna pide evidencia del uso de un asistente de IA. Documentá acá, con tus propias palabras, ejemplos reales:

| Qué le pedí a la IA | Qué me devolvió | Qué acepté / qué corregí y por qué |
|---|---|---|
| *(ej.: "¿cómo detecto duplicados por máquina y mes?")* | | |
| | | |

Guardá también capturas o el enlace de la conversación en `docs/ia_copiloto.md`.